# Семинар 2. Построение n-граммной языковой модели

**Курс:** «Интеллектуальный анализ текстов: от статистических моделей языка до языковых агентов». Семинар относится к Лекции 1 «Введение в курс. Статистическое моделирование языка».

## Цель

Реализовать n-граммную языковую модель со сглаживанием и научиться сравнивать модели по перплексии и качеству генерации.

## Задачи семинара

1. Реализовать биграммную и триграммную модель с MLE-оценками.
2. Реализовать сглаживание Лапласа и Кнесера—Нея (упрощённый вариант).
3. Посчитать перплексию на отложенной выборке для разных $n$ и схем сглаживания.
4. Сгенерировать текст top-k сэмплированием (k=8) из обученной модели.

## Ход занятия

1. Разбиение корпуса на train/dev/test.
2. Реализация подсчёта n-грамм (`Counter`/`defaultdict`) и MLE.
3. Имплементация сглаживаний, сравнение формул на бумаге и в коде.
4. Расчёт перплексии на dev, подбор гиперпараметров ($\delta$, $D$).
5. Генерация продолжений текста и обсуждение деградаций (повторы, обрывы).


## Краткая теория

### 1. Языковая модель и n-граммное приближение

**Языковая модель** — распределение вероятностей $P(w_1, \dots, w_T)$ на последовательностях токенов. По цепному правилу вероятности:
$$
P(w_1,\dots,w_T) = \prod_{t=1}^{T} P(w_t \mid w_1,\dots,w_{t-1}).
$$

**N-граммная модель** делает марковское приближение порядка $n-1$:
$$
P(w_t \mid w_1,\dots,w_{t-1}) \approx P(w_t \mid w_{t-n+1},\dots,w_{t-1}).
$$

Оценка максимального правдоподобия (MLE) по корпусу:
$$
\hat P_{\text{MLE}}(w_t \mid w_{t-n+1}^{t-1}) = \frac{C(w_{t-n+1}^{t-1}, w_t)}{C(w_{t-n+1}^{t-1})},
$$
где $C(\cdot)$ — частота n-граммы в корпусе. Чем длиннее контекст $n-1$, тем точнее модель в теории, но тем более разреженной становится статистика: большинство n-грамм из тестовых данных просто не встретится в обучающих (нулевая вероятность).

### 2. Сглаживание Лапласа (Лидстоуна)

Аддитивное сглаживание добавляет псевдосчёт $\delta$ к каждой n-грамме:
$$
\hat P_{\text{Lap}}(w_t \mid w_{t-n+1}^{t-1}) = \frac{C(w_{t-n+1}^{t-1}, w_t) + \delta}{C(w_{t-n+1}^{t-1}) + \delta |V|}.
$$
Мы «одолжили» немного вероятностной массы у часто встречающихся n-грамм в пользу невстреченных, гарантируя ненулевую вероятность для любого продолжения. При $\delta=1$ — классическое сглаживание Лапласа; при $\delta<1$ — сглаживание Лидстоуна (обычно даёт лучшую перплексию).

### 3. Линейная интерполяция моделей разного порядка

Чтобы одновременно использовать надёжность низкого порядка и точность высокого, применяют интерполяцию (для триграммной модели):
$$
P_{\text{interp}}(w_t \mid w_{t-2}, w_{t-1}) = \lambda_3 \hat P(w_t \mid w_{t-2}, w_{t-1}) + \lambda_2 \hat P(w_t \mid w_{t-1}) + \lambda_1 \hat P(w_t),
$$
при $\lambda_1+\lambda_2+\lambda_3=1,\ \lambda_i \ge 0$. Веса $\lambda_i$ подбираются на dev-выборке (в этом семинаре — простым перебором).

### 4. Сглаживание Кнесера—Нея (упрощённый вариант)

Модель использует абсолютное дисконтирование $D$ и интерполяцию с распределением по числу уникальных контекстов продолжения:
$$
P_{\text{KN}}(w_t \mid w_{t-1}) = \frac{\max(C(w_{t-1}, w_t) - D, 0)}{C(w_{t-1})} + \lambda(w_{t-1}) \, P_{\text{cont}}(w_t),
$$
где $P_{\text{cont}}(w_t) \propto |\{w' : C(w', w_t) > 0\}|$ — доля уникальных контекстов, в которых встречается $w_t$ (а не сама частота слова), и
$$
\lambda(w_{t-1}) = \frac{D \cdot |\{w' : C(w_{t-1}, w') > 0\}|}{C(w_{t-1})}
$$
— вес, обеспечивающий нормировку распределения в единицу. Интуиция: слово вроде «Francisco» частотно, но встречается почти исключительно после «San» — модель должна учитывать не саму частоту слова, а разнообразие контекстов, в которых оно продолжает n-грамму.

### 5. Перплексия

Для тестовой последовательности длины $T$:
$$
\mathrm{PPL} = \exp\left(-\frac{1}{T}\sum_{t=1}^{T}\log P(w_t \mid w_{1}^{t-1})\right).
$$
Перплексия — это «эффективный размер словаря», из которого модель в среднем выбирает следующее слово; чем ниже перплексия, тем увереннее и точнее предсказания модели. Перплексия униграммной модели с равновероятным распределением на словаре размера $|V|$ равна ровно $|V|$ (baseline для сравнения, см. также задачу 1 для самостоятельного решения).


## Подготовка

Используем тот же источник корпуса, что и в Семинаре 1: небольшой сэмпл русских новостей **Lenta.ru** через `datasets` (HuggingFace Hub, потоковый режим) с offline-фолбэком. Разбиваем предложения на train/dev/test (80/10/10), строим словарь и заменяем редкие слова на `<unk>`, чтобы модель могла корректно обрабатывать неизвестные слова.


In [ ]:
import re
import math
import random
import warnings
from collections import Counter, defaultdict

import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N_DOCS = 400        # небольшой сэмпл -- работает на CPU за секунды/десятки секунд
MIN_COUNT = 2        # слова с частотой < MIN_COUNT заменяются на <unk>
BOS, EOS, UNK = "<s>", "</s>", "<unk>"

print("Импорты готовы, SEED =", SEED)


In [ ]:
def load_lenta_sentences(n_docs=N_DOCS):
    """Загружает n_docs новостных текстов Lenta.ru через HuggingFace datasets (streaming),
    разбивает на предложения. При ошибке -- offline fallback."""
    try:
        from datasets import load_dataset
        ds = load_dataset("zloelias/lenta-ru", split="train", streaming=True)
        texts = []
        for i, row in enumerate(ds):
            if i >= n_docs:
                break
            t = (row.get("text") or "").strip()
            if t:
                texts.append(t)
        if len(texts) >= 50:
            sentences = []
            for t in texts:
                # простое разбиение на предложения по знакам конца предложения
                for s in re.split(r"(?<=[.!?])\s+", t):
                    s = s.strip()
                    if len(s.split()) >= 4:  # отбрасываем слишком короткие "предложения"
                        sentences.append(s)
            if len(sentences) >= 200:
                return sentences, "lenta_ru (HuggingFace, streaming)"
    except Exception as e:
        print("Не удалось загрузить Lenta.ru с Hub:", repr(e))

    # ---- offline fallback: небольшой встроенный корпус русских новостных предложений ----
    fallback_sentences = [
        "Правительство объявило о новых мерах поддержки малого бизнеса в этом году",
        "Центральный банк снизил ключевую ставку на фоне замедления инфляции",
        "Учёные представили новое исследование о влиянии климата на урожайность",
        "Компания сообщила о рекордной прибыли по итогам последнего квартала",
        "Власти города утвердили план реконструкции центральных улиц",
        "Спортивная сборная одержала победу в решающем матче турнира",
        "Министерство науки анонсировало запуск новой образовательной программы",
        "Аналитики прогнозируют рост цен на энергоносители в следующем месяце",
        "Разработчики выпустили обновление популярного приложения для смартфонов",
        "Депутаты приняли закон о регулировании цифровых финансовых активов",
        "Врачи предупреждают о росте заболеваемости в осенний период",
        "Инвесторы проявляют интерес к акциям технологических компаний",
        "Городская администрация запустила программу благоустройства парков",
        "Эксперты обсудили перспективы развития искусственного интеллекта",
        "Автопроизводитель представил новую модель электромобиля на выставке",
        "Пограничная служба сообщила об усилении контроля на границе",
        "Судебное разбирательство по делу о мошенничестве продолжается второй месяц",
        "Фермеры собрали рекордный урожай зерновых в этом сезоне",
        "Банк запустил новый сервис онлайн-кредитования для предпринимателей",
        "Журналисты выяснили подробности сделки между крупными корпорациями",
        "Сегодня на рынке акций наблюдается высокая волатильность торгов",
        "Аналитики отмечают рост спроса на продукцию отечественных производителей",
    ]
    random.shuffle(fallback_sentences)
    sentences = []
    for i in range(max(300, n_docs)):
        sentences.append(fallback_sentences[i % len(fallback_sentences)])
    return sentences, "offline fallback corpus (встроенные новостные предложения)"


all_sentences, corpus_source = load_lenta_sentences(N_DOCS)
random.shuffle(all_sentences)
print(f"Источник корпуса: {corpus_source}")
print(f"Всего предложений: {len(all_sentences)}")
print("Пример:", all_sentences[0])


In [ ]:
TOKEN_RE = re.compile(r"[а-яёa-z0-9]+", re.UNICODE)

def simple_tokenize(sentence):
    return TOKEN_RE.findall(sentence.lower())

tokenized_sentences = [simple_tokenize(s) for s in all_sentences]
tokenized_sentences = [s for s in tokenized_sentences if len(s) >= 3]

n = len(tokenized_sentences)
n_train = int(n * 0.8)
n_dev = int(n * 0.1)

train_sents = tokenized_sentences[:n_train]
dev_sents = tokenized_sentences[n_train:n_train + n_dev]
test_sents = tokenized_sentences[n_train + n_dev:]

print(f"train: {len(train_sents)}  dev: {len(dev_sents)}  test: {len(test_sents)}")
print("Пример токенизированного предложения:", train_sents[0])


In [ ]:
# Строим словарь по train-выборке, редкие слова -> <unk>
word_counts = Counter(w for s in train_sents for w in s)
vocab_words = {w for w, c in word_counts.items() if c >= MIN_COUNT}
vocab = set(vocab_words) | {BOS, EOS, UNK}
V_SIZE = len(vocab)

def apply_unk(sentence):
    return [w if w in vocab_words else UNK for w in sentence]

def add_boundaries(sentence, n_boundaries=1):
    """Добавляет n_boundaries токенов <s> в начало и один </s> в конец (для n-грамм порядка n_boundaries+1)."""
    return [BOS] * n_boundaries + sentence + [EOS]

train_sents_unk = [apply_unk(s) for s in train_sents]
dev_sents_unk = [apply_unk(s) for s in dev_sents]
test_sents_unk = [apply_unk(s) for s in test_sents]

print(f"Размер словаря (train, freq>={MIN_COUNT}, + спец.токены): {V_SIZE}")
print(f"Доля <unk> в train: {sum(w == UNK for s in train_sents_unk for w in s) / sum(len(s) for s in train_sents_unk):.3%}")


## Демо-задача 1. Baseline: униграммная модель

**Постановка.** Для всех последующих сравнений baseline-моделью служит **униграммная языковая модель** — модель нулевого порядка, оценивающая $P(w_t)$ без учёта контекста (MLE по частотам слов в train, со сглаживанием Лапласа $\delta=1$, чтобы избежать нулевых вероятностей на dev/test).

**Метрика.** Перплексия на dev-выборке (формула — см. теорию).


In [ ]:
def unigram_counts(sentences):
    c = Counter()
    for s in sentences:
        for w in s:
            c[w] += 1
    return c

uni_counts = unigram_counts(train_sents_unk)
total_unigrams = sum(uni_counts.values())

def unigram_logprob_laplace(word, delta=1.0):
    return math.log((uni_counts.get(word, 0) + delta) / (total_unigrams + delta * V_SIZE))

def perplexity_unigram(sentences, delta=1.0):
    """Перплексия униграммной модели (с граничными токенами </s>, без <s> -- контекст не нужен)."""
    log_prob_sum = 0.0
    n_tokens = 0
    for s in sentences:
        seq = s + [EOS]
        for w in seq:
            log_prob_sum += unigram_logprob_laplace(w, delta)
            n_tokens += 1
    return math.exp(-log_prob_sum / n_tokens)

ppl_unigram_dev = perplexity_unigram(dev_sents_unk)
print(f"Перплексия униграммной модели (baseline) на dev: {ppl_unigram_dev:.2f}")
print(f"Размер словаря |V| = {V_SIZE} (перплексия РАВНОМЕРНОЙ модели равна |V|;")
print("у модели хуже равномерной перплексия может быть и выше |V|)")


**Вывод.** Перплексия униграммной модели существенно ниже $|V|$ — перплексии равномерной модели (заметим: $|V|$ — это не верхняя граница вообще, а именно перплексия равномерного распределения; модель, оценивающая вероятности хуже равномерной, может иметь перплексию выше $|V|$), так как частоты слов сильно неравномерны (закон Ципфа, см. Семинар 1) — модель уже «знает», что частые слова («и», «в», «на») встречаются намного чаще редких. Тем не менее униграммная модель не учитывает контекст вообще, поэтому её качество генерации крайне низкое — это и есть базовый уровень, который должны превзойти биграммная и триграммная модели.


## Демо-задача 2. MLE-оценки биграммной и триграммной моделей

**Постановка.** Реализуем подсчёт n-грамм через `defaultdict(Counter)` и MLE-оценку вероятностей (без сглаживания). Посчитаем перплексию на dev — и увидим фундаментальную проблему нулевых вероятностей.

**Основная модель.** Биграммная ($n=2$) и триграммная ($n=3$) модели с MLE.

**Метрика.** Перплексия (формула в теории). Ожидается, что чистый MLE даст **бесконечную** (или очень большую) перплексию на dev из-за неувиденных n-грамм — это мотивирует переход к сглаживанию в следующей демо-задаче.


In [ ]:
def count_ngrams(sentences, n):
    """n=2 -> биграммы, n=3 -> триграммы. Возвращает context_counts, ngram_counts."""
    context_counts = defaultdict(int)
    ngram_counts = defaultdict(int)
    for s in sentences:
        seq = add_boundaries(s, n_boundaries=n - 1)
        for i in range(n - 1, len(seq)):
            context = tuple(seq[i - n + 1:i])
            word = seq[i]
            context_counts[context] += 1
            ngram_counts[(context, word)] += 1
    return context_counts, ngram_counts


bigram_ctx_counts, bigram_counts = count_ngrams(train_sents_unk, 2)
trigram_ctx_counts, trigram_counts = count_ngrams(train_sents_unk, 3)

print(f"Число уникальных биграммных контекстов:  {len(bigram_ctx_counts)}")
print(f"Число уникальных биграмм:                {len(bigram_counts)}")
print(f"Число уникальных триграммных контекстов: {len(trigram_ctx_counts)}")
print(f"Число уникальных триграмм:                {len(trigram_counts)}")


In [ ]:
def mle_logprob(context, word, ctx_counts, ngram_counts):
    c_context = ctx_counts.get(context, 0)
    if c_context == 0:
        return None  # контекст не встречался вообще
    c_ngram = ngram_counts.get((context, word), 0)
    if c_ngram == 0:
        return None  # нулевая вероятность -- MLE не определён (log(0))
    return math.log(c_ngram / c_context)


def perplexity_mle(sentences, n, ctx_counts, ngram_counts):
    """Возвращает (perplexity, доля токенов с неопределённой/нулевой MLE-вероятностью)."""
    log_prob_sum = 0.0
    n_tokens = 0
    n_undefined = 0
    for s in sentences:
        seq = add_boundaries(s, n_boundaries=n - 1)
        for i in range(n - 1, len(seq)):
            context = tuple(seq[i - n + 1:i])
            word = seq[i]
            lp = mle_logprob(context, word, ctx_counts, ngram_counts)
            if lp is None:
                n_undefined += 1
            else:
                log_prob_sum += lp
            n_tokens += 1
    frac_undefined = n_undefined / n_tokens
    if n_undefined > 0:
        return float("inf"), frac_undefined
    return math.exp(-log_prob_sum / n_tokens), frac_undefined


ppl_bigram_mle, frac_undef_bi = perplexity_mle(dev_sents_unk, 2, bigram_ctx_counts, bigram_counts)
ppl_trigram_mle, frac_undef_tri = perplexity_mle(dev_sents_unk, 3, trigram_ctx_counts, trigram_counts)

print(f"Биграммная MLE:  перплексия = {ppl_bigram_mle}, доля неопределённых токенов = {frac_undef_bi:.1%}")
print(f"Триграммная MLE: перплексия = {ppl_trigram_mle}, доля неопределённых токенов = {frac_undef_tri:.1%}")


**Таблица сравнения с baseline.**

| Модель | Перплексия на dev | Комментарий |
|---|---|---|
| Униграммная (baseline) | конечное значение (см. вывод демо-задачи 1) | ненулевая вероятность для любого слова словаря |
| Биграммная MLE (без сглаживания) | $\infty$ (при наличии неувиденных биграмм) | нулевая вероятность на части dev — модель «ломается» |
| Триграммная MLE (без сглаживания) | $\infty$, доля неопределённых токенов выше, чем у биграммной | разреженность растёт с порядком n-граммы |

**Вывод.** Чистая MLE-оценка биграммной и тем более триграммной модели даёт бесконечную перплексию на отложенной выборке, так как некоторые контексты/n-граммы dev не встречались в train ни разу — это именно та проблема разреженности данных, о которой говорится в лекции. Единственный baseline с конечной перплексией — униграммная модель (она способна оценить вероятность любого слова из словаря). Это мотивирует переход к сглаживанию (следующая демо-задача).


## Демо-задача 3. Сглаживания: Лаплас, интерполяция, обзорно Кнесер—Ней

**Постановка.** Реализуем сглаживание Лапласа для биграммной модели, линейную интерполяцию биграмм с униграммами и упрощённый вариант сглаживания Кнесера—Нея. Сравним перплексию на dev.

**Baseline.** Униграммная модель из демо-задачи 1.

**Метрика.** Перплексия на dev (формула — см. теорию).


In [ ]:
def bigram_logprob_laplace(context, word, delta=1.0):
    c_context = bigram_ctx_counts.get(context, 0)
    c_ngram = bigram_counts.get((context, word), 0)
    return math.log((c_ngram + delta) / (c_context + delta * V_SIZE))


def perplexity_bigram_laplace(sentences, delta=1.0):
    log_prob_sum = 0.0
    n_tokens = 0
    for s in sentences:
        seq = add_boundaries(s, n_boundaries=1)
        for i in range(1, len(seq)):
            context = (seq[i - 1],)
            word = seq[i]
            log_prob_sum += bigram_logprob_laplace(context, word, delta)
            n_tokens += 1
    return math.exp(-log_prob_sum / n_tokens)


ppl_bigram_laplace = perplexity_bigram_laplace(dev_sents_unk, delta=1.0)
print(f"Биграммная модель + Лаплас (delta=1.0): перплексия на dev = {ppl_bigram_laplace:.2f}")


In [ ]:
def bigram_logprob_interp(context, word, lam2, uni_delta=1.0):
    """Линейная интерполяция: lam2 * P_bigram_MLE + (1-lam2) * P_unigram_laplace."""
    c_context = bigram_ctx_counts.get(context, 0)
    p_bi = (bigram_counts.get((context, word), 0) / c_context) if c_context > 0 else 0.0
    p_uni = math.exp(unigram_logprob_laplace(word, delta=uni_delta))
    p = lam2 * p_bi + (1 - lam2) * p_uni
    return math.log(p) if p > 0 else float("-inf")


def perplexity_bigram_interp(sentences, lam2):
    log_prob_sum = 0.0
    n_tokens = 0
    for s in sentences:
        seq = add_boundaries(s, n_boundaries=1)
        for i in range(1, len(seq)):
            context = (seq[i - 1],)
            word = seq[i]
            log_prob_sum += bigram_logprob_interp(context, word, lam2)
            n_tokens += 1
    return math.exp(-log_prob_sum / n_tokens)


# Подбор lambda2 на dev небольшим перебором (заготовка для задачи по Лапласу -- аналогичный подход)
best_lam2, best_ppl_interp = None, float("inf")
for lam2 in [0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    ppl = perplexity_bigram_interp(dev_sents_unk, lam2)
    if ppl < best_ppl_interp:
        best_lam2, best_ppl_interp = lam2, ppl

print(f"Лучшая lambda2 = {best_lam2}, перплексия интерполированной модели на dev = {best_ppl_interp:.2f}")


In [ ]:
def build_continuation_counts(ngram_counts_bi):
    """P_cont(w) ~ число уникальных левых контекстов, после которых встречается w."""
    continuation_sets = defaultdict(set)
    for (context, word) in ngram_counts_bi:
        continuation_sets[word].add(context)
    total_unique_pairs = sum(len(v) for v in continuation_sets.values())
    p_cont = {w: len(ctxs) / total_unique_pairs for w, ctxs in continuation_sets.items()}
    return p_cont

p_cont = build_continuation_counts(bigram_counts)

# Наивный перебор всех биграмм на каждый вызов был бы медленным, поэтому заранее
# строим словарь context -> число уникальных продолжений
context_n_continuations = defaultdict(int)
for (ctx, w) in bigram_counts:
    context_n_continuations[ctx] += 1

def bigram_logprob_kneser_ney_fast(context, word, D=0.75):
    """Упрощённый Кнесер-Ней для биграмм (абсолютное дисконтирование + continuation prob).

    NB (упрощение): для слов, не встречавшихся в биграммах train, p_cont неопределена,
    и мы подставляем 1/V (p_cont.get(word, 1.0 / V_SIZE)). Это удобная заглушка, но она
    НЕ гарантирует, что итоговое распределение по словарю нормировано ровно в единицу;
    полноценный Кнесер-Ней требует аккуратной нормировки continuation-распределения
    по всему словарю (включая unseen-слова)."""
    c_context = bigram_ctx_counts.get(context, 0)
    if c_context == 0:
        p = p_cont.get(word, 1.0 / V_SIZE)
        return math.log(max(p, 1e-12))
    c_ngram = bigram_counts.get((context, word), 0)
    discounted = max(c_ngram - D, 0) / c_context
    n_continuations = context_n_continuations.get(context, 0)
    lam = (D * n_continuations) / c_context
    p = discounted + lam * p_cont.get(word, 1.0 / V_SIZE)
    return math.log(max(p, 1e-12))

def perplexity_bigram_kn_fast(sentences, D=0.75):
    log_prob_sum = 0.0
    n_tokens = 0
    for s in sentences:
        seq = add_boundaries(s, n_boundaries=1)
        for i in range(1, len(seq)):
            context = (seq[i - 1],)
            word = seq[i]
            log_prob_sum += bigram_logprob_kneser_ney_fast(context, word, D)
            n_tokens += 1
    return math.exp(-log_prob_sum / n_tokens)

ppl_bigram_kn = perplexity_bigram_kn_fast(dev_sents_unk, D=0.75)
print(f"Биграммная модель + Кнесер-Ней (D=0.75, упрощённый): перплексия на dev = {ppl_bigram_kn:.2f}")


**Таблица сравнения всех схем сглаживания (биграммная модель, dev).**

| Модель | Перплексия на dev | Идея сглаживания |
|---|---|---|
| Униграммная (baseline) | см. демо-задача 1 | сглаживание Лапласа по словам, без контекста |
| Биграммы + Лаплас ($\delta=1$) | см. вывод кода выше | равномерная добавка псевдосчёта |
| Биграммы + интерполяция с униграммами | см. вывод кода выше (лучшая $\lambda_2$) | взвешенное смешивание с надёжной моделью низкого порядка |
| Биграммы + Кнесер—Ней (упрощ., $D=0.75$) | см. вывод кода выше | дисконтирование + учёт разнообразия контекстов продолжения |

**Вывод.** Простое сглаживание Лапласа снижает перплексию с бесконечности до конечного, но обычно довольно большого значения, так как равномерно «размазывает» вероятностную массу по всему словарю. Интерполяция с униграммной моделью и особенно упрощённый Кнесер—Ней обычно дают заметно более низкую перплексию, так как учитывают более тонкую структуру данных (надёжность контекста низкого порядка и разнообразие продолжений).


## Демо-задача 4. Генерация текста top-k сэмплированием (k=8)

**Постановка.** Сгенерируем продолжения текста, сэмплируя следующее слово из распределения биграммной модели со сглаживанием Кнесера—Нея (основная модель) и сравним с генерацией из **baseline** — униграммной модели (сэмплирование без учёта контекста).

**Метрика.** Качественная оценка: доля уникальных слов в сгенерированном тексте (простая мера отсутствия деградации/повторов) — чем выше, тем меньше «застревания» в циклах.


In [ ]:
def sample_from_dist(word_probs, rng):
    words, probs = zip(*word_probs.items())
    probs = np.array(probs)
    probs = probs / probs.sum()
    return rng.choice(words, p=probs)


def generate_unigram(rng, max_len=12):
    """Baseline: генерация из униграммного распределения (без учёта контекста)."""
    candidates = {w: (c + 1.0) for w, c in uni_counts.items() if w not in (BOS, EOS)}
    words = []
    for _ in range(max_len):
        w = sample_from_dist(candidates, rng)
        if w == EOS:
            break
        words.append(w)
    return " ".join(words)


def generate_bigram_kn(rng, max_len=12, D=0.75, top_k=8):
    """Основная модель: сэмплирование из top_k наиболее вероятных продолжений биграммной
    модели со сглаживанием Кнесера-Нея (top-k сэмплирование для устойчивости; не путать с жадным декодированием, k=1)."""
    context = BOS
    words = []
    vocab_list = list(vocab_words) + [EOS]
    for _ in range(max_len):
        scored = []
        for w in vocab_list:
            lp = bigram_logprob_kneser_ney_fast((context,), w, D)
            scored.append((lp, w))
        scored.sort(reverse=True)
        top = scored[:top_k]
        probs = {w: math.exp(lp) for lp, w in top}
        w_next = sample_from_dist(probs, rng)
        if w_next == EOS:
            break
        words.append(w_next)
        context = w_next
    return " ".join(words)


rng = np.random.default_rng(SEED)

print("=== Генерация: baseline (униграммная модель) ===")
for _ in range(3):
    print(" -", generate_unigram(rng))

print("\n=== Генерация: биграммная модель + Кнесер-Ней ===")
for _ in range(3):
    print(" -", generate_bigram_kn(rng))


In [ ]:
def unique_word_ratio(text):
    words = text.split()
    return len(set(words)) / len(words) if words else 0.0

uni_samples = [generate_unigram(rng) for _ in range(5)]
kn_samples = [generate_bigram_kn(rng) for _ in range(5)]

ratio_uni = np.mean([unique_word_ratio(t) for t in uni_samples])
ratio_kn = np.mean([unique_word_ratio(t) for t in kn_samples])

print(f"{'Модель':<35}{'Средняя доля уникальных слов':>30}")
print(f"{'Униграммная (baseline)':<35}{ratio_uni:>30.2f}")
print(f"{'Биграммная + Кнесер-Ней':<35}{ratio_kn:>30.2f}")


**Вывод.** Генерация из униграммной модели даёт синтаксически несвязный «поток слов» — слова выбираются независимо от контекста, поэтому итоговый текст не образует осмысленных словосочетаний. Биграммная модель с Кнесером—Неем учитывает предыдущее слово и генерирует более связные локальные словосочетания, однако на маленьком корпусе легко «застревает» в часто встречающихся парах слов (артефакт коротких циклов) — типичная деградация n-граммных моделей, которую в лекции 5 будет решать переход к нейронным языковым моделям.


## Задачи для самостоятельного решения

Ниже — 5 задач из списка задач семинара в силлабусе (адаптированные под данный корпус и переменные, уже созданные выше). Для каждой задачи: посчитайте метрику, сравните с указанным baseline, сделайте вывод (1–2 предложения) в отдельной markdown-ячейке после кода.


### Задача 1. Сравнение перплексии биграммной и триграммной моделей (со сглаживанием Лапласа)

Постройте триграммную модель со сглаживанием Лапласа (аналогично `bigram_logprob_laplace`, но для контекста длины 2, используя `trigram_ctx_counts`/`trigram_counts`) и сравните перплексию на dev с биграммной моделью + Лаплас (`ppl_bigram_laplace`, посчитанной в демо-задаче 3 — это **baseline** для сравнения).

**Ожидаемый диапазон результата:** на небольшом корпусе триграммная модель с простым Лапласом часто получает перплексию **выше** (хуже), чем биграммная, — из-за более сильной разреженности триграмм; это нормальный и ожидаемый результат, который стоит объяснить в выводе.


In [ ]:
# TODO: реализуйте trigram_logprob_laplace(context, word, delta=1.0) по аналогии с bigram_logprob_laplace
#       (контекст -- тьюпл из 2 слов, используйте trigram_ctx_counts и trigram_counts)
# TODO: реализуйте perplexity_trigram_laplace(sentences, delta=1.0)
# TODO: посчитайте ppl_trigram_laplace = perplexity_trigram_laplace(dev_sents_unk, delta=1.0)
# TODO: сравните ppl_trigram_laplace с ppl_bigram_laplace (baseline, из демо-задачи 3)
# TODO: сделайте вывод в markdown-ячейке ниже

ppl_trigram_laplace = None  # замените на вычисленное значение


**Вывод:** _(заполните после выполнения кода выше)_

### Задача 2. Подбор оптимального $\delta$ для сглаживания Лапласа

Постройте график зависимости перплексии биграммной модели (`perplexity_bigram_laplace`) от параметра $\delta \in \{0.001, 0.01, 0.1, 0.5, 1.0, 2.0\}$ на dev-выборке и найдите оптимум. Сравните минимальную найденную перплексию с перплексией при $\delta=1$ (классический Лаплас, посчитанный в демо-задаче 3 — это **baseline**).

**Ожидаемый диапазон результата:** оптимальное $\delta$ обычно значительно меньше 1 (в диапазоне 0.01–0.5) и даёт перплексию заметно ниже, чем при $\delta=1$, — избыточное сглаживание при $\delta=1$ на малом словаре пере-выравнивает вероятности.


In [ ]:
# TODO: для каждого delta из [0.001, 0.01, 0.1, 0.5, 1.0, 2.0] вычислите
#       ppl = perplexity_bigram_laplace(dev_sents_unk, delta=delta)
# TODO: постройте график perplexity(delta) (можно в полулогарифмической шкале по delta)
# TODO: найдите delta с минимальной перплексией, сравните с ppl_bigram_laplace при delta=1.0 (baseline)
# TODO: сделайте вывод в markdown-ячейке ниже


**Вывод:** _(заполните после выполнения кода выше)_

### Задача 3. Генерация продолжений фразы «Сегодня на рынке»

Сгенерируйте 5 продолжений фразы «Сегодня на рынке» биграммной моделью с Кнесером—Неем (функция `generate_bigram_kn`, но со стартовым контекстом не `<s>`, а последним словом фразы) и сравните разнообразие (`unique_word_ratio`) с генерацией из униграммной модели (**baseline**, `generate_unigram`) для того же числа продолжений.

**Ожидаемый диапазон результата:** биграммные продолжения обычно содержат заметные артефакты — короткие циклы («застревание» на паре часто соседствующих слов), особенно на маленьком корпусе; доля уникальных слов в них может быть даже **ниже**, чем у случайного униграммного baseline, — это ожидаемый и важный для обсуждения результат.


In [ ]:
# TODO: модифицируйте generate_bigram_kn (или напишите новую функцию), чтобы генерация
#       начиналась с context = "рынке" (последнее слово фразы "Сегодня на рынке")
# TODO: сгенерируйте 5 продолжений, посчитайте unique_word_ratio для каждого и среднее
# TODO: сравните со средним unique_word_ratio для 5 генераций generate_unigram (baseline)
# TODO: сделайте вывод о наблюдаемых артефактах (циклы, повторы) в markdown-ячейке ниже


**Вывод:** _(заполните после выполнения кода выше)_

### Задача 4. Проверка: перплексия равновероятной униграммной модели равна $|V|$

Постройте искусственную («игрушечную») равновероятную униграммную модель: $P(w) = 1/|V|$ для каждого слова словаря (без учёта реальных частот) и эмпирически проверьте на dev, что её перплексия равна $|V|$ (с точностью до вычислительной ошибки). Сравните с перплексией настоящей униграммной модели с MLE+Лапласом из демо-задачи 1 (**baseline**, `ppl_unigram_dev`).

**Ожидаемый диапазон результата:** перплексия равновероятной модели должна совпасть с `V_SIZE` почти точно (отличие < 0.01% из-за округления); перплексия настоящей униграммной модели должна быть заметно **ниже** `V_SIZE`, так как она использует статистику частот слов.


In [ ]:
# TODO: реализуйте perplexity_uniform(sentences) -- аналог perplexity_unigram,
#       но log P(w) = -log(V_SIZE) для любого слова (равновероятная модель)
# TODO: посчитайте ppl_uniform = perplexity_uniform(dev_sents_unk)
# TODO: сравните ppl_uniform с V_SIZE (должны почти совпасть) и с ppl_unigram_dev (baseline)
# TODO: сделайте вывод в markdown-ячейке ниже

ppl_uniform = None  # замените на вычисленное значение


**Вывод:** _(заполните после выполнения кода выше)_

### Задача 5. Сглаживание Гуда—Тьюринга на редких биграммах (сравнение с Лапласом)

Сравним два метода на одном языке — через **скорректированную частоту** $r^*$: сколько «эффективных» вхождений метод оставляет биграмме, встретившейся в train ровно $r$ раз.

- **Гуд—Тьюринг:** $r^* = (r+1)\dfrac{N_{r+1}}{N_r}$, где $N_r$ — число различных биграмм, встретившихся ровно $r$ раз. Считайте $N_r$ по `bigram_counts` для $r \le 6$: на больших $r$ статистика шумная.
- **Лаплас (baseline, $\delta = 1$)** как одно распределение по всем $B = |V|^2$ возможным биграммам: $r^*_{\text{Lap}} = (r+1)\dfrac{N}{N + B}$, где $N$ — общее число биграмм в train (сумма `bigram_counts.values()`), $|V|$ = `len(vocab)`. Это частота после сглаживания Лапласа, пересчитанная из вероятности обратно во «вхождения».

Сравните $r^*$ обоих методов для $r = 1, 2, 3$ и долю вероятностной массы, которую каждый метод отдаёт **невиданным** биграммам: $N_1/N$ у Гуда—Тьюринга и $(B - T)/(N + B)$ у Лапласа, где $T$ = `len(bigram_counts)` — число различных биграмм в train.

**Ожидаемый диапазон результата:** у Гуда—Тьюринга $r^*$ меньше $r$, но того же порядка (для $r = 1$ примерно 0.2–0.6); невиданным достаётся около 30–50% массы. Лаплас при $|V|^2$ порядка $10^7$ сжимает наблюдённые частоты в сотни раз ($r^*_{\text{Lap}}$ порядка $10^{-3}$–$10^{-2}$) и отдаёт невиданным больше 99% массы — это и есть «грубость» Лапласа, о которой шла речь на лекции.

**Важно:** задача требует настоящего корпуса Lenta.ru. На встроенном офлайн-корпусе (несколько десятков предложений, повторённых по кругу) каждая биграмма встречается много раз, все $N_r$ для малых $r$ равны нулю, и формула Гуда—Тьюринга не определена.


In [ ]:
# TODO: посчитайте N_r -- число различных биграмм, встретившихся ровно r раз, для r=1..6 (по bigram_counts.values())
# TODO: Гуд-Тьюринг: r* = (r+1) * N_{r+1} / N_r для r=1,2,3
# TODO: Лаплас (baseline, delta=1): r*_lap = (r+1) * N / (N + B), где N = sum(bigram_counts.values()), B = len(vocab)**2
# TODO: масса на невиданные биграммы: N_1 / N (Гуд-Тьюринг) и (B - T) / (N + B) (Лаплас), T = len(bigram_counts)
# TODO: сведите результаты в таблицу и сделайте вывод в markdown-ячейке ниже

Nr = {}  # заполните: Nr[r] = количество биграмм с частотой r

**Вывод:** _(заполните после выполнения кода выше)_

## Чек-лист оценивания

При проверке решений семинара учитывайте следующие критерии:

- [ ] **Корректное разбиение данных**: корпус разделён на train/dev/test без пересечений; словарь построен только по train, редкие слова заменены на `<unk>`.
- [ ] **Наличие baseline**: для каждой задачи есть явное сравнение с базовым решением (униграммная модель, сглаживание Лапласа с $\delta=1$, равновероятное распределение и т.п.).
- [ ] **Выбор метрики**: используется перплексия с явной формулой; для генерации — дополнительная качественная метрика (доля уникальных слов).
- [ ] **Воспроизводимость**: зафиксирован `random.seed`/`np.random.seed`/`np.random.default_rng`; результаты повторяемы при повторном запуске.
- [ ] **Вывод**: для каждой демо-задачи и каждой самостоятельной задачи есть содержательный вывод (1–2 предложения), интерпретирующий полученные числа (а не просто их пересказ), в том числе объяснение неожиданных результатов (например, рост перплексии триграммной модели относительно биграммной на малом корпусе).
